# 4. Phylogeny

Understanding the evolutionary relationships between microbial taxa is a key component of microbiome analysis. Phylogenetic trees provide a framework for interpreting community structure, measuring diversity, and linking taxonomic groups to functional or ecological patterns. 
In this notebook, we generate a de novo phylogenetic tree. This means that all ASVs in our dataset are placed within a consistent evolutionary framework without relying on external reference databases. Building a phylogeny from 16S RNA requires first aligning our sequences and then building the phylogeny that represents the distances between these sequences.
To assess the reliability of the inferred tree topology, we then perform bootstrapping, a resampling-based approach that quantifies confidence in each clade. Bootstrapping tests how consistently individual branches appear across many pseudo-replicated datasets, allowing us to identify well-supported versus uncertain regions of the tree.
The final phylogenetic tree can be used for downstream analyses such as UniFrac distance calculation or phylogenetic diversity metrics.


### Notebook Structure

**1.** De novo tree construction  
&nbsp;&nbsp;&nbsp;&nbsp;**1.1** Sequence Alignment   
&nbsp;&nbsp;&nbsp;&nbsp;**1.2** Alignment Masking   
&nbsp;&nbsp;&nbsp;&nbsp;**1.3** Tree Construction  
&nbsp;&nbsp;&nbsp;&nbsp;**1.4** Tree Rooting   
&nbsp;&nbsp;&nbsp;&nbsp;**1.5** Tree Visualization   
&nbsp;&nbsp;&nbsp;&nbsp;**1.6** Bootstrapping   
**2.** Fragment insertion  
&nbsp;&nbsp;&nbsp;&nbsp;**2.1** Silva Reference  
&nbsp;&nbsp;&nbsp;&nbsp;**2.2** Tree Construction using SEPP  
&nbsp;&nbsp;&nbsp;&nbsp;**2.3** Tree Visualization  


In [1]:
# 1- Import packages
import os
import pandas as pd
from qiime2 import Visualization
import matplotlib.pyplot as plt
import numpy as np
import qiime2 as q2

%matplotlib inline

We are using empress for the tree visualization. Since it is not part of qiime, we need to install the package via pip first.

In [2]:
!pip install empress

  Using cached empress-1.2.0-py3-none-any.whl
  Using cached nose-1.3.7-py3-none-any.whl.metadata (1.7 kB)
  Using cached cython-3.2.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached scikit_bio-0.5.9-cp310-cp310-linux_x86_64.whl
  Using cached scipy-1.10.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (58 kB)
Using cached scipy-1.10.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (34.4 MB)
Using cached cython-3.2.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (3.5 MB)
Using cached nose-1.3.7-py3-none-any.whl (154 kB)
  Attempting uninstall: scipy
    Found existing installation: scipy 1.13.0
    Uninstalling scipy-1.13.0:
      Successfully uninstalled scipy-1.13.0━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [scipy]
  Attempting uninstall: scikit-bio━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [cython]
    Found existing installation: scikit-bio 0.6.2━━━━━━━━━━━━━ 2/5 [cython]
    Uni

In [3]:
import empress

/opt/conda/lib/python3.10/site-packages/empress/core.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# 2 - Set working directory
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/Project/MicrobiomeAnalysis_TummyTribe/scripts").


In [5]:
# 3 - Data directories
meta_data_dir = "../data/processed/metadata"
raw_data_dir = "../data/raw"
phylogeny_data_dir = "../data/processed/phylogeny"
denoising_data_dir = "../data/processed/denoising"
taxonomy_data_dir = "../data/processed/taxonomy"

In [6]:
%%bash -s "$phylogeny_data_dir"
mkdir -p "$1"

## 1. De novo tree construction
We can create a phylogenetic tree by aligning the marker genes across divergent taxa and try to reconstruct the tree based on the resulting alignment. One of the issues of this approach is that short sequences may not carry enough information to capture a meaningful phylogeny. Generally fragment insertion is recommended for 16S rRNA data, but since our goal is to study evolutionary patterns and diversity within our cohort, it makes sense to use de novo tree construction [Price et al. (2010)]. Fragment insertion is described as an alternative method further down in this notebook but will not be used in our analysis.

Standard pipeline for de novo tree construction:
1. MAFFT multiple-sequence alignment
2. Masking (filtering) the alignment
3. FastTree (or RAxML/IQ-TREE) tree inference
4. Midpoint rooting

After MAFFT aligns the ASVs, QIIME masks the alignment to remove hypervariable, gappy, or uninformative positions that would distort phylogenetic inference. The masked alignment is then passed to FastTree, which uses an approximate maximum-likelihood algorithm (GTR+CAT model) to rapidly construct a de novo phylogeny optimized for large microbial datasets.

### 1.1 Sequence alignemnt

In [10]:
! qiime alignment mafft \
    --i-sequences $denoising_data_dir/dada2_rep_seq.qza \
    --o-alignment $phylogeny_data_dir/aligned-rep-seqs.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[AlignedSequence] to: ../data/processed/phylogeny/aligned-rep-seqs.qza


### 1.2 Alignment masking

In [11]:
! qiime alignment mask \
    --i-alignment $phylogeny_data_dir/aligned-rep-seqs.qza \
    --o-masked-alignment $phylogeny_data_dir/masked-aligned-rep-seqs.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved FeatureData[AlignedSequence] to: ../data/processed/phylogeny/masked-aligned-rep-seqs.qza


### 1.3 Tree inference

In [12]:
! qiime phylogeny fasttree \
    --i-alignment $phylogeny_data_dir/masked-aligned-rep-seqs.qza \
    --o-tree $phylogeny_data_dir/fasttree-tree.qza

! qiime phylogeny midpoint-root \
    --i-tree $phylogeny_data_dir/fasttree-tree.qza \
    --o-rooted-tree $phylogeny_data_dir/fasttree-tree-rooted.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Phylogeny[Unrooted] to: ../data/processed/phylogeny/fasttree-tree.qza
/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Phylogeny[Rooted] to: ../data/processed/phylogeny/fasttree-tree-rooted.qza


### 1.4 Tree rooting

In [13]:
! qiime empress tree-plot \
    --i-tree $phylogeny_data_dir/fasttree-tree-rooted.qza \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --o-visualization $phylogeny_data_dir/fasttree-tree-rooted.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/fasttree-tree-rooted.qzv


### 1.5 Tree viszualization

In [16]:
! qiime empress community-plot \
    --i-tree $phylogeny_data_dir/fasttree-tree-rooted.qza \
    --i-feature-table $denoising_data_dir/dada2_table.qza \
    --m-sample-metadata-file $meta_data_dir/metadata_merged.tsv \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --p-filter-extra-samples \
    --o-visualization $phylogeny_data_dir/fasttree-community-tree-viz.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/fasttree-community-tree-viz.qzv


In [17]:
Visualization.load(f"{phylogeny_data_dir}/fasttree-community-tree-viz.qzv")

<visualization: Visualization uuid: 0c2a8645-ec36-4967-8701-af75469c18c4>

In the interactive tree visualization we suggest to change the layout to rectangular or circular to see the relationships better. Additionally you can color by Feature Metadata at level 6 (genus level). You can also try to color by Sample Metadata. 

### 1.5 Bootstrapping
Bootstrapping provides confidence values for each branch and helps identify which parts of the tree are well supported versus uncertain. This is done by taking random subsamples of the dataset, building trees from each of these and calculating the frequency with which the various parts of our de novo tree are reproduced in each of these random subsamples [Baldauf. (2003)].

It is important to mention that bootstrapping can't help with any of the shortcomings of a de novo tree. Bootstrapping measures confidence, but does not improve accuracy. A weak or biased de novo tree remains weak or biased, even with high bootstrap support.

This cell does run in the notebook, but you might want to get a coffee (or two) since it takes about 4 hours. 

In [4]:
! qiime phylogeny raxml-rapid-bootstrap \
    --i-alignment $phylogeny_data_dir/masked-aligned-rep-seqs.qza \
    --p-seed 1723 \
    --p-rapid-bootstrap-seed 9384 \
    --p-bootstrap-replicates 100 \
    --p-substitution-model GTRCAT \
    --p-n-threads 3 \
    --o-tree $phylogeny_data_dir/raxml-cat-bootstrap-tree.qza

IndentationError: unexpected indent (438486707.py, line 2)

In [6]:
! qiime phylogeny midpoint-root \
    --i-tree $phylogeny_data_dir/raxml-cat-bootstrap-tree.qza \
    --o-rooted-tree $phylogeny_data_dir/raxml-cat-bootstrap-tree-rooted.qza

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Phylogeny[Rooted] to: ../data/processed/phylogeny/raxml-cat-bootstrap-tree-rooted.qza


In [8]:
! qiime empress community-plot \
    --i-tree $phylogeny_data_dir/raxml-cat-bootstrap-tree-rooted.qza \
    --i-feature-table $denoising_data_dir/dada2_table.qza \
    --m-sample-metadata-file $meta_data_dir/metadata_merged.tsv \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --p-filter-extra-samples \
    --o-visualization $phylogeny_data_dir/raxml-cat-bootstrap-tree.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/raxml-cat-bootstrap-tree.qzv


In [18]:
Visualization.load(f"{phylogeny_data_dir}/raxml-cat-bootstrap-tree.qzv")

<visualization: Visualization uuid: d18b77c0-007e-4b6f-9bc3-1b0540f164a5>

Comparing the original de novo tree with the tree after bootstrapping, they look very similar, which is what we would expect. We will use the bootstrap tree for any downstream analysis.

## 2. Fragment insertion

Fragment insertion uses a reference tree and then our sequences are inserted into this tree. This results in a way larger tree than if we construct a de novo tree since it includes the entire reference backbone tree. We can get the SILVA 128 reference tree from qiime. According to a user in the qiime forum using SILVA 138 for the taxonomy and SILVA 128 to build the phylogenetic tree should not lead to any complications (https://forum.qiime2.org/t/compatibility-of-sepp-silva128-with-taxonomy-classification-using-silva138/31172?utm_source=chatgpt.com).

For our analysis we have decided to use the de novo tree since it is easier to construct and works well with the dataset that we are using.

### 2.1 Silva Reference 

In [18]:
#SILVA 128 for SEPP
! wget -O $phylogeny_data_dir/silva-128-sepp-refs.qza https://data.qiime2.org/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza

--2025-11-11 13:28:36--  https://data.qiime2.org/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza
Resolving data.qiime2.org (data.qiime2.org)... 54.200.1.12
Connecting to data.qiime2.org (data.qiime2.org)|54.200.1.12|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza [following]
--2025-11-11 13:28:36--  https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sepp-ref-dbs/sepp-refs-silva-128.qza
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.218.246.176, 52.92.162.192, 52.218.236.160, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.218.246.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 181253322 (173M) [binary/octet-stream]
Saving to: ‘../data/processed/phylogeny/silva-128-sepp-refs.qza’

../data/processed/p 100%[===================>] 172.86M  16.3MB/s    in 12s     

2025-11-11

### 2.2 Tree construction using SEPP
The following cell does not run in this notebook due to limited memory. It was run on euler and the finished tree and tree placements can be imported for visualization and further analysis.

In [21]:
#does not run in notebook -> euler (needs amplicon env)
! qiime fragment-insertion sepp \
    --i-representative-sequences $denoising_data_dir/dada2_rep_seq.qza \
    --i-reference-database $phylogeny_data_dir/silva-128-sepp-refs.qza \
    --p-threads 2 \
    --o-tree $phylogeny_data_dir/sepp-tree.qza \
    --o-placements $phylogeny_data_dir/sepp-tree-placements.qza \
    --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Removing /tmp/tmp.1ZZbpa3ZS5/sepp-tmp-QfUBleakIt


Check if outpoot tree has expected format. It's already rooted and has Newick tree format.

### 2.3 Tree visualization

In [9]:
! qiime empress tree-plot \
    --i-tree $phylogeny_data_dir/sepp-tree.qza \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --o-visualization $phylogeny_data_dir/sepp-tree.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/sepp-tree.qzv


In [16]:
! qiime empress community-plot \
    --i-tree $phylogeny_data_dir/sepp-tree.qza \
    --i-feature-table $denoising_data_dir/dada2_table.qza \
    --m-sample-metadata-file $metadata_dir/metadata_merged.tsv \
    --m-feature-metadata-file $taxonomy_data_dir/taxonomy.qza \
    --p-filter-extra-samples \
    --o-visualization $phylogeny_data_dir/sepp-community-tree-viz-filtered.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/phylogeny/sepp-community-tree-viz-filtered.qzv


In [17]:
Visualization.load(f"{phylogeny_data_dir}/sepp-community-tree-viz-filtered.qzv")

<visualization: Visualization uuid: 048bd4e2-0461-4045-b58e-e03f60de5ca8>

## References

Price MN, Dehal PS, Arkin AP. FastTree 2--approximately maximum-likelihood trees for large alignments. PLoS One. (2010). 

Baldauf SL. Phylogeny for the faint of heart: a tutorial. Trends Genet. (2003).